<a href="https://colab.research.google.com/github/asha-murthy/WeatherForecast/blob/main/TerraBlue_XT_DataScientist_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The Problem

Weather forecasts are systematically biased. A forecast model might consistently over-predict
wind speed in valleys and under-predict it on ridgelines. Your job is to build a model that learns
and corrects this bias.

# Fetching the data

Picking 10 locations across  different terrain types

Fetching hourly data for each location: wind speed (m/s) and wind direction from both
APIs, for the period 1 January 2025 to 31 December 2025.

In [13]:
!pip install openmeteo-requests
!pip install requests-cache retry-requests numpy pandas

In [14]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry


cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

url_forecast = "https://historical-forecast-api.open-meteo.com/v1/forecast"
params_forecast = {
	"latitude": [8, 25, 22, 27, 32, 3, 42, 20, 36, 2],
	"longitude": [76, 55, 43, 86, 70, 37, 103, 66, 117, 59],
	"start_date": "2025-01-01",
	"end_date": "2025-12-31",
	"hourly": ["wind_speed_10m", "wind_direction_10m"],
	"wind_speed_unit": "ms",
}
responses_forecast = openmeteo.weather_api(url_forecast, params = params_forecast)

forecast_dataframes = []
# Process 10 locations
for response in responses_forecast:
	# Process hourly data. The order of variables needs to be the same as requested.
	hourly = response.Hourly()
	hourly_wind_speed_10m = hourly.Variables(0).ValuesAsNumpy()
	hourly_wind_direction_10m = hourly.Variables(1).ValuesAsNumpy()

	hourly_data = {
		"date": pd.date_range(
			start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
			end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
			freq = pd.Timedelta(seconds = hourly.Interval()),
			inclusive = "left"
		)
	}

	hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
	hourly_data["wind_direction_10m"] = hourly_wind_direction_10m

	location_df = pd.DataFrame(data = hourly_data)
	location_df['latitude'] = response.Latitude()
	location_df['longitude'] = response.Longitude()
	forecast_dataframes.append(location_df)


In [15]:
forecast_dataframes

[                          date  wind_speed_10m  wind_direction_10m  latitude  \
 0    2025-01-01 00:00:00+00:00        7.213875           43.876797       8.0   
 1    2025-01-01 01:00:00+00:00        7.433034           42.273628       8.0   
 2    2025-01-01 02:00:00+00:00        7.789737           41.877792       8.0   
 3    2025-01-01 03:00:00+00:00        7.874643           40.364468       8.0   
 4    2025-01-01 04:00:00+00:00        8.060397           37.438648       8.0   
 ...                        ...             ...                 ...       ...   
 8755 2025-12-31 19:00:00+00:00        3.671512          299.357666       8.0   
 8756 2025-12-31 20:00:00+00:00        3.764306          309.610657       8.0   
 8757 2025-12-31 21:00:00+00:00        4.101220          315.000092       8.0   
 8758 2025-12-31 22:00:00+00:00        4.404543          320.527557       8.0   
 8759 2025-12-31 23:00:00+00:00        4.280187          322.594574       8.0   
 
       longitude  
 0     

In [16]:
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url_archive = "https://archive-api.open-meteo.com/v1/archive"
params_archive = {
	"latitude": [8, 25, 22, 27, 32, 3, 42, 20, 36, 2],
	"longitude": [76, 55, 43, 86, 70, 37, 103, 66, 117, 59],
	"start_date": "2025-01-01",
	"end_date": "2025-12-31",
	"hourly": ["wind_speed_10m", "wind_direction_10m"],
	"models": "era5",
	"wind_speed_unit": "ms",
}
responses_archive = openmeteo.weather_api(url_archive, params = params_archive)

historical_dataframes = []
# Process 10 locations
for response in responses_archive:
	# Process hourly data. The order of variables needs to be the same as requested.
	hourly = response.Hourly()
	hourly_wind_speed_10m = hourly.Variables(0).ValuesAsNumpy()
	hourly_wind_direction_10m = hourly.Variables(1).ValuesAsNumpy()

	hourly_data = {
		"date": pd.date_range(
			start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
			end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
			freq = pd.Timedelta(seconds = hourly.Interval()),
			inclusive = "left"
		)
	}

	hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
	hourly_data["wind_direction_10m"] = hourly_wind_direction_10m

	location_df = pd.DataFrame(data = hourly_data)
	location_df['latitude'] = response.Latitude()
	location_df['longitude'] = response.Longitude()
	historical_dataframes.append(location_df)

In [17]:
historical_dataframes

[                          date  wind_speed_10m  wind_direction_10m  latitude  \
 0    2025-01-01 00:00:00+00:00        4.443254           39.062580       8.0   
 1    2025-01-01 01:00:00+00:00        4.974434           39.289394       8.0   
 2    2025-01-01 02:00:00+00:00        5.550901           35.837746       8.0   
 3    2025-01-01 03:00:00+00:00        6.161371           38.078938       8.0   
 4    2025-01-01 04:00:00+00:00        6.616079           42.856228       8.0   
 ...                        ...             ...                 ...       ...   
 8755 2025-12-31 19:00:00+00:00        4.078296          310.525024       8.0   
 8756 2025-12-31 20:00:00+00:00        4.389191          312.229736       8.0   
 8757 2025-12-31 21:00:00+00:00        4.702393          315.430725       8.0   
 8758 2025-12-31 22:00:00+00:00        5.215362          327.528839       8.0   
 8759 2025-12-31 23:00:00+00:00        5.308719          330.054413       8.0   
 
       longitude  
 0     

In [18]:
combined_df = pd.concat(forecast_dataframes, ignore_index=True)
combined_df['type'] = 'forecast'

archive_df = pd.concat(historical_dataframes, ignore_index=True)
archive_df['type'] = 'archive'

all_data = pd.concat([combined_df, archive_df], ignore_index=True)

print("Combined DataFrame head:")
print(all_data.head())
print("\nCombined DataFrame info:")
print(all_data.info())
print("\nValue counts for 'type' column:")
print(all_data['type'].value_counts())

Combined DataFrame head:
                       date  wind_speed_10m  wind_direction_10m  latitude  \
0 2025-01-01 00:00:00+00:00        7.213875           43.876797       8.0   
1 2025-01-01 01:00:00+00:00        7.433034           42.273628       8.0   
2 2025-01-01 02:00:00+00:00        7.789737           41.877792       8.0   
3 2025-01-01 03:00:00+00:00        7.874643           40.364468       8.0   
4 2025-01-01 04:00:00+00:00        8.060397           37.438648       8.0   

   longitude      type  
0       76.0  forecast  
1       76.0  forecast  
2       76.0  forecast  
3       76.0  forecast  
4       76.0  forecast  

Combined DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 175200 entries, 0 to 175199
Data columns (total 6 columns):
 #   Column              Non-Null Count   Dtype              
---  ------              --------------   -----              
 0   date                175200 non-null  datetime64[ns, UTC]
 1   wind_speed_10m      175200 non-null

# Feature Engineering

In [19]:
all_data['hour'] = all_data['date'].dt.hour
all_data['day_of_week'] = all_data['date'].dt.dayofweek
all_data['day_of_year'] = all_data['date'].dt.dayofyear
all_data['month'] = all_data['date'].dt.month

print("DataFrame after feature engineering:")
print(all_data.head())

DataFrame after feature engineering:
                       date  wind_speed_10m  wind_direction_10m  latitude  \
0 2025-01-01 00:00:00+00:00        7.213875           43.876797       8.0   
1 2025-01-01 01:00:00+00:00        7.433034           42.273628       8.0   
2 2025-01-01 02:00:00+00:00        7.789737           41.877792       8.0   
3 2025-01-01 03:00:00+00:00        7.874643           40.364468       8.0   
4 2025-01-01 04:00:00+00:00        8.060397           37.438648       8.0   

   longitude      type  hour  day_of_week  day_of_year  month  
0       76.0  forecast     0            2            1      1  
1       76.0  forecast     1            2            1      1  
2       76.0  forecast     2            2            1      1  
3       76.0  forecast     3            2            1      1  
4       76.0  forecast     4            2            1      1  


# Preparing Data for Bias Modeling

In [20]:
pivoted_df_speed = all_data.pivot_table(
    index=['date', 'latitude', 'longitude', 'hour', 'day_of_week', 'day_of_year', 'month'],
    columns='type',
    values='wind_speed_10m'
).reset_index()
pivoted_df_speed.rename(columns={'forecast': 'wind_speed_10m_forecast', 'archive': 'wind_speed_10m_archive'}, inplace=True)

pivoted_df_direction = all_data.pivot_table(
    index=['date', 'latitude', 'longitude', 'hour', 'day_of_week', 'day_of_year', 'month'],
    columns='type',
    values='wind_direction_10m'
).reset_index()
pivoted_df_direction.rename(columns={'forecast': 'wind_direction_10m_forecast', 'archive': 'wind_direction_10m_archive'}, inplace=True)

# Merge the pivoted dataframes
bias_data = pd.merge(pivoted_df_speed, pivoted_df_direction,
                     on=['date', 'latitude', 'longitude', 'hour', 'day_of_week', 'day_of_year', 'month'])

# Calculate the bias
bias_data['wind_speed_10m_bias'] = bias_data['wind_speed_10m_forecast'] - bias_data['wind_speed_10m_archive']
bias_data['wind_direction_10m_bias'] = bias_data['wind_direction_10m_forecast'] - bias_data['wind_direction_10m_archive']

print("Bias Data DataFrame head:")
print(bias_data.head())
print("\nBias Data DataFrame info:")
print(bias_data.info())


Bias Data DataFrame head:
type                      date  latitude  longitude  hour  day_of_week  \
0    2025-01-01 00:00:00+00:00       2.0       59.0     0            2   
1    2025-01-01 00:00:00+00:00       3.0       37.0     0            2   
2    2025-01-01 00:00:00+00:00       8.0       76.0     0            2   
3    2025-01-01 00:00:00+00:00      20.0       66.0     0            2   
4    2025-01-01 00:00:00+00:00      22.0       43.0     0            2   

type  day_of_year  month  wind_speed_10m_archive  wind_speed_10m_forecast  \
0               1      1                5.331041                 4.252059   
1               1      1                2.516446                 3.764306   
2               1      1                4.443254                 7.213875   
3               1      1                3.244996                 3.023243   
4               1      1                3.763642                 3.138471   

type  wind_direction_10m_archive  wind_direction_10m_forecast  \
0

# Handling Missing Bias Values

In [21]:
# Drop rows with missing values in the bias columns
bias_data.dropna(subset=['wind_speed_10m_bias', 'wind_direction_10m_bias'], inplace=True)

print("Bias Data DataFrame head after dropping NaNs:")
print(bias_data.head())
print("\nBias Data DataFrame info after dropping NaNs:")
print(bias_data.info())

Bias Data DataFrame head after dropping NaNs:
type                      date  latitude  longitude  hour  day_of_week  \
0    2025-01-01 00:00:00+00:00       2.0       59.0     0            2   
1    2025-01-01 00:00:00+00:00       3.0       37.0     0            2   
2    2025-01-01 00:00:00+00:00       8.0       76.0     0            2   
3    2025-01-01 00:00:00+00:00      20.0       66.0     0            2   
4    2025-01-01 00:00:00+00:00      22.0       43.0     0            2   

type  day_of_year  month  wind_speed_10m_archive  wind_speed_10m_forecast  \
0               1      1                5.331041                 4.252059   
1               1      1                2.516446                 3.764306   
2               1      1                4.443254                 7.213875   
3               1      1                3.244996                 3.023243   
4               1      1                3.763642                 3.138471   

type  wind_direction_10m_archive  wind_directi

# Preparing Data for Model Training
Defining Features (X) and Target Variables (y)

Splitting the Data

In [22]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Define features (X) and target variables (y)
features = ['latitude', 'longitude', 'hour', 'day_of_week', 'day_of_year', 'month', 'wind_speed_10m_forecast', 'wind_direction_10m_forecast']
target_speed_bias = 'wind_speed_10m_bias'
target_direction_bias = 'wind_direction_10m_bias'

X = bias_data[features]
y_speed = bias_data[target_speed_bias]
y_direction = bias_data[target_direction_bias]

# Split data into training and testing sets based on months
# Training: January to October (months 1-10)
# Testing: November to December (months 11-12)

X_train = X[X['month'] <= 10]
y_speed_train = y_speed[X['month'] <= 10]
y_direction_train = y_direction[X['month'] <= 10]

X_test = X[X['month'] > 10]
y_speed_test = y_speed[X['month'] > 10]
y_direction_test = y_direction[X['month'] > 10]

print(f"Training set size: {len(X_train)} samples")
print(f"Test set size: {len(X_test)} samples")


gbr_speed = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gbr_speed.fit(X_train, y_speed_train)

gbr_direction = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gbr_direction.fit(X_train, y_direction_train)

print("\nGradient Boosting models trained successfully!")

Training set size: 65664 samples
Test set size: 13176 samples

Gradient Boosting models trained successfully!


# Model Evaluation

In [23]:
# Predict on the test set for wind speed bias
y_speed_pred = gbr_speed.predict(X_test)

# Evaluate wind speed bias model
mae_speed = mean_absolute_error(y_speed_test, y_speed_pred)
mse_speed = mean_squared_error(y_speed_test, y_speed_pred)
rmse_speed = np.sqrt(mse_speed)

print(f"\nWind Speed Bias Model Evaluation:")
print(f"  Mean Absolute Error (MAE): {mae_speed:.4f}")
print(f"  Mean Squared Error (MSE): {mse_speed:.4f}")
print(f"  Root Mean Squared Error (RMSE): {rmse_speed:.4f}")

# Predict on the test set for wind direction bias
y_direction_pred = gbr_direction.predict(X_test)

# Evaluate wind direction bias model
mae_direction = mean_absolute_error(y_direction_test, y_direction_pred)
mse_direction = mean_squared_error(y_direction_test, y_direction_pred)
rmse_direction = np.sqrt(mse_direction)

print(f"\nWind Direction Bias Model Evaluation:")
print(f"  Mean Absolute Error (MAE): {mae_direction:.4f}")
print(f"  Mean Squared Error (MSE): {mse_direction:.4f}")
print(f"  Root Mean Squared Error (RMSE): {rmse_direction:.4f}")


Wind Speed Bias Model Evaluation:
  Mean Absolute Error (MAE): 0.6081
  Mean Squared Error (MSE): 0.6636
  Root Mean Squared Error (RMSE): 0.8146

Wind Direction Bias Model Evaluation:
  Mean Absolute Error (MAE): 39.2458
  Mean Squared Error (MSE): 4622.7517
  Root Mean Squared Error (RMSE): 67.9908


# Applying Bias Correction

Corrected Forecast = Original Forecast - Predicted Bias

In [24]:
# Get the original forecast values from the test set
original_speed_forecast = X_test['wind_speed_10m_forecast']
original_direction_forecast = X_test['wind_direction_10m_forecast']

# Get the actual archive values from the test set
actual_speed = bias_data[bias_data['month'] > 10]['wind_speed_10m_archive']
actual_direction = bias_data[bias_data['month'] > 10]['wind_direction_10m_archive']

# Predict the bias for the test set
predicted_speed_bias = gbr_speed.predict(X_test)
predicted_direction_bias = gbr_direction.predict(X_test)

# Apply the bias correction: Corrected Forecast = Original Forecast - Predicted Bias
corrected_speed_forecast = original_speed_forecast - predicted_speed_bias
corrected_direction_forecast = original_direction_forecast - predicted_direction_bias

# Evaluate the corrected forecasts
mae_corrected_speed = mean_absolute_error(actual_speed, corrected_speed_forecast)
mse_corrected_speed = mean_squared_error(actual_speed, corrected_speed_forecast)
rmse_corrected_speed = np.sqrt(mse_corrected_speed)

print(f"\nCorrected Wind Speed Forecast Evaluation:")
print(f"  Mean Absolute Error (MAE): {mae_corrected_speed:.4f}")
print(f"  Mean Squared Error (MSE): {mse_corrected_speed:.4f}")
print(f"  Root Mean Squared Error (RMSE): {rmse_corrected_speed:.4f}")

mae_corrected_direction = mean_absolute_error(actual_direction, corrected_direction_forecast)
mse_corrected_direction = mean_squared_error(actual_direction, corrected_direction_forecast)
rmse_corrected_direction = np.sqrt(mse_corrected_direction)

print(f"\nCorrected Wind Direction Forecast Evaluation:")
print(f"  Mean Absolute Error (MAE): {mae_corrected_direction:.4f}")
print(f"  Mean Squared Error (MSE): {mse_corrected_direction:.4f}")
print(f"  Root Mean Squared Error (RMSE): {rmse_corrected_direction:.4f}")



Corrected Wind Speed Forecast Evaluation:
  Mean Absolute Error (MAE): 0.6081
  Mean Squared Error (MSE): 0.6636
  Root Mean Squared Error (RMSE): 0.8146

Corrected Wind Direction Forecast Evaluation:
  Mean Absolute Error (MAE): 39.2458
  Mean Squared Error (MSE): 4622.7517
  Root Mean Squared Error (RMSE): 67.9908
